Sequences store items by position. Dictionaries store items by name — you look up a value by a key rather than by an index. Sets store items with no duplicates and no order, and are optimised for membership testing and mathematical set operations. Both are implemented as hash tables, which makes lookup, insertion, and deletion essentially constant time regardless of how large the collection grows. This notebook covers dictionaries and sets from the basics through to the collections module helpers that make common patterns more expressive.

## Dictionaries

A dictionary maps **keys** to **values**. You look up a value by providing its key — just like looking up a word in a physical dictionary.

- Keys must be **hashable** (immutable): strings, numbers, tuples of hashable items. Lists cannot be keys.
- Values can be anything — any Python object.
- Keys are **unique** — assigning to an existing key replaces its value.
- As of Python 3.7, dictionaries maintain **insertion order**.

In [ ]:
# Dictionary literal — curly braces, key: value pairs
person = {
    "name":  "Alice",
    "age":   30,
    "city":  "Tokyo",
    "active": True,
}

# dict() constructor from keyword arguments
config = dict(host="localhost", port=5432, debug=False)

# dict() from a list of (key, value) pairs
codes = dict([("USD", "Dollar"), ("EUR", "Euro"), ("JPY", "Yen")])

print(person)
print(config)
print(codes)

### Accessing and Modifying Values

In [ ]:
person = {"name": "Alice", "age": 30, "city": "Tokyo"}

# Access by key — raises KeyError if key is missing
print(person["name"])   # Alice

# .get() — returns None (or a default) if key is missing, never raises
print(person.get("age"))          # 30
print(person.get("email"))        # None
print(person.get("email", "N/A")) # N/A — custom default

# Add a new key-value pair
person["email"] = "alice@example.com"

# Update an existing value
person["age"] = 31

print(person)

In [ ]:
person = {"name": "Alice", "age": 30, "city": "Tokyo", "email": "alice@example.com"}

# del — remove a key (raises KeyError if absent)
del person["email"]

# pop() — remove and return a value (optional default avoids KeyError)
age = person.pop("age")
print(f"Removed age: {age}")

missing = person.pop("phone", None)   # no KeyError
print(f"phone: {missing}")

print(person)

### Iterating Over Dictionaries

In [ ]:
config = {"host": "localhost", "port": 5432, "debug": False}

# Iterating directly gives keys
for key in config:
    print(key)

print()

# .keys(), .values(), .items() — all return view objects that reflect live changes
print(list(config.keys()))    # ['host', 'port', 'debug']
print(list(config.values()))  # ['localhost', 5432, False]

# .items() is the most useful — yields (key, value) pairs
for key, value in config.items():
    print(f"  {key} = {value}")

In [ ]:
# Membership testing — 'in' checks keys
print("host" in config)     # True
print("timeout" in config)  # False

# To check values, use 'in config.values()'
print(5432 in config.values())  # True

### Merging and Updating Dictionaries

In [ ]:
defaults = {"timeout": 30, "retries": 3, "debug": False}
overrides = {"debug": True, "host": "prod.example.com"}

# .update() — merges another dict in place (right dict wins on conflicts)
merged = defaults.copy()
merged.update(overrides)
print(merged)

# | operator (Python 3.9+) — returns a new merged dict
merged = defaults | overrides
print(merged)

# |= operator (Python 3.9+) — in-place merge
defaults |= overrides
print(defaults)

### Useful Dictionary Methods

In [ ]:
inventory = {"apples": 5, "bananas": 3}

# setdefault — return existing value, or insert a default and return it
# Does NOT overwrite if the key already exists
count = inventory.setdefault("apples", 0)    # key exists → returns 5, no change
print(count, inventory)

count = inventory.setdefault("cherries", 0)  # new key → inserts 0, returns 0
print(count, inventory)

# Clear all items
temp = {"a": 1, "b": 2}
temp.clear()
print(temp)   # {}

# Copy
original = {"x": 10}
copy = original.copy()
copy["y"] = 20
print(original, copy)

### Dictionary Comprehensions

Just as list comprehensions build lists concisely, dictionary comprehensions build dicts. The syntax mirrors list comprehensions but uses curly braces and a `key: value` expression.

In [ ]:
# Square each number
squares = {n: n ** 2 for n in range(1, 8)}
print(squares)   # {1: 1, 2: 4, 3: 9, 4: 16, 5: 25, 6: 36, 7: 49}

# Invert a dictionary (swap keys and values)
codes = {"USD": "Dollar", "EUR": "Euro", "JPY": "Yen"}
inverted = {v: k for k, v in codes.items()}
print(inverted)

# Filter — keep only passing students
scores = {"Alice": 88, "Bob": 54, "Carol": 76, "Dave": 45}
passing = {name: score for name, score in scores.items() if score >= 60}
print(passing)

### Nested Dictionaries

Dictionary values can themselves be dictionaries. This is the natural way to represent structured, hierarchical data — configuration files, API responses, user records.

In [ ]:
users = {
    "alice": {"age": 30, "role": "admin",  "scores": [95, 88, 92]},
    "bob":   {"age": 25, "role": "viewer", "scores": [70, 65, 80]},
}

# Access nested values by chaining []
print(users["alice"]["role"])       # admin
print(users["bob"]["scores"][1])    # 65

# Safely navigate with .get() to avoid KeyError on missing keys
email = users.get("alice", {}).get("email", "not set")
print(email)   # not set

# Add a new field to a nested dict
users["alice"]["email"] = "alice@example.com"

# Iterate nested structure
for username, info in users.items():
    avg = sum(info["scores"]) / len(info["scores"])
    print(f"{username}: role={info['role']}, avg score={avg:.1f}")

### `defaultdict` and `Counter`

The `collections` module provides two dictionary subclasses that handle very common patterns:

- **`defaultdict`**: like a regular dict, but automatically inserts a default value when a missing key is accessed — no more `setdefault` or `if key not in d` guards.
- **`Counter`**: a dict subclass designed for counting. Pass it any iterable and it tallies the occurrences of each item.

In [ ]:
from collections import defaultdict

# Group words by their first letter
words = ["apple", "avocado", "banana", "blueberry", "cherry", "apricot"]

# Without defaultdict — needs an explicit guard
groups = {}
for word in words:
    key = word[0]
    if key not in groups:
        groups[key] = []
    groups[key].append(word)

# With defaultdict — automatically creates an empty list for new keys
groups = defaultdict(list)
for word in words:
    groups[word[0]].append(word)   # no guard needed

for letter, group in sorted(groups.items()):
    print(f"{letter}: {group}")

In [ ]:
from collections import Counter

text = "the quick brown fox jumps over the lazy dog the end"
word_counts = Counter(text.split())

print(word_counts)
print(word_counts.most_common(3))   # three most frequent words
print(word_counts["the"])           # 3
print(word_counts["cat"])           # 0 — missing keys return 0, not KeyError

# Counter works on any iterable — characters, list items, etc.
letter_freq = Counter("mississippi")
print(letter_freq.most_common())    # sorted by frequency

## Sets

A set is an **unordered** collection of **unique** items. Duplicates are silently discarded. Sets are implemented as hash tables, so membership testing (`in`) is O(1) — as fast for a million-item set as for a ten-item set.

Use a set when:
- You need to eliminate duplicates from a collection.
- You need fast membership testing.
- You need to compute unions, intersections, or differences between collections.

In [ ]:
# Set literal — curly braces, no colon (unlike dict)
fruits = {"apple", "banana", "cherry", "apple", "banana"}
print(fruits)   # {'apple', 'banana', 'cherry'} — duplicates removed, order not guaranteed

# set() constructor — converts any iterable, deduplifying as it goes
nums = set([3, 1, 4, 1, 5, 9, 2, 6, 5, 3])
print(nums)

unique_chars = set("mississippi")
print(unique_chars)   # {'m', 'i', 's', 'p'}

# Empty set — MUST use set(), not {} (which creates an empty dict)
empty_set  = set()
empty_dict = {}    # dict, not set!
print(type(empty_set), type(empty_dict))

### Adding and Removing Items

In [ ]:
tags = {"python", "data", "ml"}

# add — adds one item (no effect if already present)
tags.add("ai")
tags.add("python")   # already in set — silently ignored
print(tags)

# update — adds all items from another iterable
tags.update(["cloud", "devops"])
print(tags)

# remove — raises KeyError if not found
tags.remove("ml")

# discard — same as remove but no error if missing
tags.discard("missing_tag")   # no error

# pop — removes and returns an arbitrary item
item = tags.pop()
print(f"Popped: {item}, remaining: {tags}")

### Set Operations

Sets support all the standard mathematical operations. Each has both a method and an operator form.

| Operation | Operator | Method |
|-----------|----------|--------|
| Union | `a \| b` | `a.union(b)` |
| Intersection | `a & b` | `a.intersection(b)` |
| Difference | `a - b` | `a.difference(b)` |
| Symmetric difference | `a ^ b` | `a.symmetric_difference(b)` |

In [ ]:
python_devs = {"Alice", "Bob", "Carol", "Dave"}
go_devs     = {"Bob", "Dave", "Eve", "Frank"}

# Union — everyone in either team
print(python_devs | go_devs)

# Intersection — in both teams
print(python_devs & go_devs)    # {'Bob', 'Dave'}

# Difference — in Python but not Go
print(python_devs - go_devs)    # {'Alice', 'Carol'}

# Symmetric difference — in one team but not both
print(python_devs ^ go_devs)    # {'Alice', 'Carol', 'Eve', 'Frank'}

In [ ]:
# In-place versions modify the set directly
admins = {"Alice", "Bob"}
admins |= {"Carol"}         # add Carol to admins
print(admins)

active_users = {"Alice", "Dave", "Eve"}
admins &= active_users      # keep only admins who are active
print(admins)               # {'Alice'}

### Set Comparisons

In [ ]:
a = {1, 2, 3}
b = {1, 2, 3, 4, 5}
c = {6, 7, 8}

# Subset — every item in a is also in b
print(a <= b)               # True  — a is a subset of b
print(a.issubset(b))        # True

# Proper subset — subset but not equal
print(a < b)                # True  — a is a proper subset of b
print(a < a)                # False — a is not a proper subset of itself

# Superset
print(b >= a)               # True  — b is a superset of a
print(b.issuperset(a))      # True

# Disjoint — no items in common
print(a.isdisjoint(c))      # True
print(a.isdisjoint(b))      # False

In [ ]:
# Practical: fast deduplication
log_entries = ["error", "warning", "error", "info", "error", "warning"]
unique_levels = list(set(log_entries))
print(unique_levels)   # order not guaranteed

# Preserve insertion order while deduplicating (Python 3.7+)
seen = set()
deduped = [x for x in log_entries if not (x in seen or seen.add(x))]
print(deduped)   # ['error', 'warning', 'info'] — first-occurrence order preserved

### `frozenset`

A `frozenset` is an **immutable** set. Because it is hashable, it can be used as a dictionary key or as an element of another set — something a regular (mutable) set cannot do.

In [ ]:
fs = frozenset(["read", "write", "execute"])
print(fs)
print(type(fs))

# frozenset can be a dict key
permissions = {
    frozenset(["read"]):              "viewer",
    frozenset(["read", "write"]):     "editor",
    frozenset(["read", "write", "execute"]): "admin",
}

user_perms = frozenset(["read", "write"])
print(permissions[user_perms])   # editor

# All set operations work on frozensets — they return frozensets
a = frozenset([1, 2, 3])
b = frozenset([2, 3, 4])
print(a & b)   # frozenset({2, 3})

## Summary

**Dictionaries**
- Map keys to values. Keys must be hashable; values can be any object. Keys are unique; last write wins. Insertion order is preserved (Python 3.7+).
- Access: `d[key]` (KeyError if missing) or `d.get(key, default)` (safe).
- Modify: `d[key] = value`, `del d[key]`, `d.pop(key, default)`.
- Iterate: `d.keys()`, `d.values()`, `d.items()`. Membership testing (`in`) checks keys.
- Merge: `d.update(other)`, `d | other` (new dict), `d |= other` (in place) — Python 3.9+.
- `setdefault(key, default)` — inserts default only if key is absent.
- **Dictionary comprehension**: `{k: v for k, v in iterable if condition}`
- **`defaultdict`**: auto-inserts a default when a missing key is accessed.
- **`Counter`**: counts occurrences; `.most_common(n)` returns the top n items; missing keys return 0.

**Sets**
- Unordered collection of unique hashable items. Duplicates are silently discarded.
- Create with `{a, b, c}` or `set(iterable)`. Empty set must be `set()` — `{}` is an empty dict.
- Modify: `add()`, `update()`, `remove()` (KeyError if absent), `discard()` (safe), `pop()` (arbitrary).
- Operations: `|` union, `&` intersection, `-` difference, `^` symmetric difference. In-place: `|=`, `&=`, `-=`, `^=`.
- Comparisons: `<=` subset, `<` proper subset, `>=` superset, `.isdisjoint()` no overlap.
- **`frozenset`**: immutable set — hashable, usable as a dict key or set element.